# 12 · Proyecto final: de un vídeo a datos estructurados (≈4 h)

Juntamos todo en **un pipeline completo**:

```
vídeo ─► YOLO11 (detección) ─► BoT-SORT/ByteTrack (tracking online) ─► embeddings de apariencia
      ─► limpieza + stitching ReID + interpolación (offline) ─► homografía → metros
      ─► trayectorias suavizadas, velocidades, distancias, eventos ─► Parquet/JSON + vídeo con minimapa
      ─► evaluación (TrackEval-like con GT de CVAT) + perfil de tiempos
```

**Hazlo tuyo**: pon tu vídeo deportivo en `data/sport.mp4` y los puntos de la homografía de tu campo (sección 0).
Apunta los números que salgan (FPS, nº de IDs antes/después, IDF1 si anotas GT).

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte, clona el repo y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics", "pyarrow": "pyarrow"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
if ROOT is None:   # este notebook usa código del repo (vision_utils/ o 11_produccion/) → clonarlo
    ROOT = Path.cwd() / "vision-bootcamp"
    url = "https://github.com/jcandel01/vision-bootcamp.git"
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(ROOT)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo: " + r.stderr)
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
import sys, json, time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from ultralytics import YOLO

OUT = DATA / "proyecto"; OUT.mkdir(exist_ok=True)
from vision_utils.postprocess import interpolate_tracks, stitch
from vision_utils.analytics import kinematics, detect_events, player_summary
from vision_utils.mot_metrics import evaluate
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

## 0. Configuración

**Homografía**: lista de puntos de la imagen (píxeles) y sus coordenadas reales en el suelo (metros), ≥ 4.
- Para un campo de fútbol: esquinas de las áreas, intersecciones de la línea de medio campo con las bandas,
  puntos de penalti… (medidas en el notebook 08). Lee los píxeles con la celda de ayuda de abajo.
- Para `people.mp4` dejo una **aproximación** estimada a ojo a partir de la altura de las personas (~115 px/m
  abajo, ~80 px/m arriba): suficiente para ver el flujo, no para medir en serio.

In [ ]:
VIDEO = DATA / "sport.mp4" if (DATA / "sport.mp4").exists() else DATA / "people.mp4"
WEIGHTS = DATA / "yolo11n.pt"          # prueba también yolo11s.pt / tu modelo fine-tuneado
IMGSZ = 960                            # personas pequeñas/lejanas → más resolución
MAX_FRAMES = None                      # None = todo el vídeo
CLASSES = [0]                          # personas (añade 32 = sports ball si te interesa el balón)

if VIDEO.name == "people.mp4":
    IMG_PTS = [[0, 0], [1920, 0], [1920, 1080], [0, 1080]]
    WORLD_PTS = [[0, 0], [24, 0], [20.35, 15], [3.65, 15]]     # metros (aprox.)
    WORLD_SIZE = (24, 15)
else:
    IMG_PTS = None       # ← RELLENA con ≥ 4 puntos de tu campo, p. ej. [[x1, y1], [x2, y2], ...]
    WORLD_PTS = None     # ← sus coordenadas en metros, p. ej. [[0, 0], [16.5, 13.84], ...]
    WORLD_SIZE = (105, 68)

cap = cv2.VideoCapture(str(VIDEO)); FPS = cap.get(cv2.CAP_PROP_FPS) or 25
W, H = int(cap.get(3)), int(cap.get(4)); N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ok, first = cap.read(); cap.release()
print(f"{VIDEO.name}: {W}x{H} @ {FPS:.0f} FPS, {N} frames ({N / FPS:.1f} s)")

In [ ]:
# Ayuda para leer coordenadas de píxeles: rejilla sobre el primer frame
g = first.copy()
for x in range(0, W, 100):
    cv2.line(g, (x, 0), (x, H), (0, 255, 255), 1); cv2.putText(g, str(x), (x + 2, 20), 0, 0.6, (0, 255, 255), 2)
for y in range(0, H, 100):
    cv2.line(g, (0, y), (W, y), (0, 255, 255), 1); cv2.putText(g, str(y), (2, y - 3), 0, 0.6, (0, 255, 255), 2)
if IMG_PTS:
    for p in IMG_PTS: cv2.circle(g, tuple(map(int, p)), 12, (0, 0, 255), -1)
plt.figure(figsize=(16, 9)); plt.imshow(g[:, :, ::-1]); plt.axis("off");
# Alternativa interactiva: en una celda aparte, `%matplotlib macosx` y luego  pts = plt.ginput(4, timeout=0)

## 1. Detección + tracking online (+ embeddings de apariencia)

Usamos BoT-SORT de Ultralytics con un YAML propio (detector a `conf` bajo para aprovechar detecciones ocluidas).
A la vez extraemos embeddings de apariencia de los recortes cada `EMB_EVERY` frames para el stitching offline
(ResNet18 ImageNet como extractor genérico; en producción, un modelo ReID como OSNet).
Medimos el tiempo de cada etapa.

In [ ]:
import yaml, ultralytics
base_cfg = Path(ultralytics.__file__).parent / "cfg" / "trackers" / "botsort.yaml"
cfg = yaml.safe_load(base_cfg.read_text())
cfg.update(track_high_thresh=0.4, track_low_thresh=0.1, new_track_thresh=0.5, track_buffer=60, gmc_method="none")
TRACKER_YAML = OUT / "botsort_custom.yaml"
TRACKER_YAML.write_text(yaml.safe_dump(cfg))
print(cfg)

from torchvision.models import resnet18, ResNet18_Weights
emb_net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1); emb_net.fc = nn.Identity()
emb_net = emb_net.eval().to(DEVICE)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

@torch.inference_mode()
def embed_crops(frame, boxes):
    crops = []
    for x1, y1, x2, y2 in boxes.astype(int):
        c = frame[max(y1, 0):max(y2, y1 + 2), max(x1, 0):max(x2, x1 + 2)]
        crops.append(cv2.resize(c, (64, 128))[:, :, ::-1])
    x = torch.from_numpy(np.stack(crops).copy()).to(DEVICE).permute(0, 3, 1, 2).float().div(255)
    e = emb_net((x - MEAN) / STD)
    return nn.functional.normalize(e, dim=1).cpu().numpy()

In [ ]:
EMB_EVERY = 5
model = YOLO(str(WEIGHTS))
cap = cv2.VideoCapture(str(VIDEO))
rows, embs, timing = [], {}, defaultdict(list)
f = 0
while MAX_FRAMES is None or f < MAX_FRAMES:
    t0 = time.perf_counter(); ok, frame = cap.read(); timing["decode"].append(time.perf_counter() - t0)
    if not ok: break
    f += 1
    t0 = time.perf_counter()
    r = model.track(frame, persist=True, tracker=str(TRACKER_YAML), conf=0.1, imgsz=IMGSZ, classes=CLASSES,
                    device=DEVICE, verbose=False)[0]
    timing["detect+track"].append(time.perf_counter() - t0)
    if r.boxes.id is None: continue
    b = r.boxes.xyxy.cpu().numpy(); ids = r.boxes.id.int().cpu().numpy(); sc = r.boxes.conf.cpu().numpy()
    for bb, i, s in zip(b, ids, sc):
        rows.append([f, i, *bb, s])
    if f % EMB_EVERY == 1:
        t0 = time.perf_counter()
        for i, e in zip(ids, embed_crops(frame, b)):
            embs[(f, int(i))] = e
        timing["reid"].append(time.perf_counter() - t0)
cap.release()
raw = np.array(rows)
n_frames = f
tm = pd.Series({k: 1000 * np.mean(v) for k, v in timing.items()})
print(f"{n_frames} frames, {len(raw)} cajas con ID, {len(np.unique(raw[:, 1]))} IDs")
print("ms medios por llamada:\n", tm.round(1))
print(f"→ ≈ {1000 / (tm['decode'] + tm['detect+track'] + tm.get('reid', 0) / EMB_EVERY):.1f} FPS de extremo a extremo")

## 2. Limpieza y diagnóstico de los tracks

In [ ]:
lens = pd.Series(raw[:, 1]).value_counts()
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(lens.values, bins=40); ax[0].set_title("longitud de los tracks (frames)"); ax[0].set_xlabel("frames")
per_frame = pd.Series(raw[:, 0]).value_counts().sort_index()
ax[1].plot(per_frame.index, per_frame.values); ax[1].set_title("tracks activos por frame"); plt.show()

MIN_LEN = max(5, int(0.3 * FPS))      # tracks de < 0.3 s suelen ser falsos positivos o fragmentos
keep = lens[lens >= MIN_LEN].index
clean = raw[np.isin(raw[:, 1], keep)]
print(f"IDs: {len(lens)} → {len(keep)} tras quitar tracks de < {MIN_LEN} frames")
print(f"personas visibles por frame (mediana): {int(per_frame.median())} → si hay muchos más IDs que eso, hay fragmentación")

## 3. Offline: stitching por apariencia + interpolación

Para cada fila necesitamos un embedding: usamos el más cercano en el tiempo del mismo track (los calculamos cada
`EMB_EVERY` frames). Los embeddings genéricos de ImageNet distinguen peor a las personas que un modelo ReID, así
que el umbral de apariencia es **conservador** y la restricción física (velocidad máxima en píxeles/frame) pesa.

In [ ]:
emb_rows = np.zeros((len(clean), 512), np.float32)
by_track = defaultdict(list)
for (fr, tid), e in embs.items(): by_track[tid].append((fr, e))
for k, (fr, tid) in enumerate(clean[:, :2].astype(int)):
    cand = by_track.get(tid)
    if cand:
        emb_rows[k] = min(cand, key=lambda c: abs(c[0] - fr))[1]
    else:
        emb_rows[k] = 1 / np.sqrt(512)

D_tmp = 1 - np.array([by_track[t][0][1] for t in list(by_track)[:40]]) @ np.array([by_track[t][0][1] for t in list(by_track)[:40]]).T
print(f"distancia coseno entre personas distintas (aprox.): mediana {np.median(D_tmp[np.triu_indices(len(D_tmp), 1)]):.3f}")

max_speed_px = 0.08 * H / FPS * 5      # ~ un 8% de la altura de la imagen por segundo ×5 de margen (ajústalo)
stitched, links = stitch(clean, emb_rows, max_gap=int(3 * FPS), max_speed=max_speed_px, app_thr=0.12)
final = interpolate_tracks(stitched, max_gap=int(1 * FPS))
print(f"stitching: {len(links)} uniones → {len(np.unique(clean[:, 1]))} IDs → {len(np.unique(stitched[:, 1]))} IDs")
print(f"interpolación: {len(stitched)} → {len(final)} filas")

Revisa algunas uniones a mano (*false merges* son peores que fragmentos): muestra el último recorte del track A
y el primero del B.

In [ ]:
def crop_at(track, which):
    t = clean[clean[:, 1] == track]; t = t[np.argsort(t[:, 0])]
    row = t[-1] if which == "last" else t[0]
    cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, int(row[0]) - 1); ok, fr = cap.read(); cap.release()
    x1, y1, x2, y2 = row[2:6].astype(int)
    return fr[max(y1, 0):y2, max(x1, 0):x2][:, :, ::-1], int(row[0])

pairs = list(links.items())[:6]
if pairs:
    fig, axs = plt.subplots(2, len(pairs), figsize=(3 * len(pairs), 7), squeeze=False)
    for j, (a, b) in enumerate(pairs):
        ca, fa = crop_at(a, "last"); cb, fb = crop_at(b, "first")
        axs[0, j].imshow(ca); axs[0, j].set_title(f"{a} (fin f{fa})"); axs[1, j].imshow(cb); axs[1, j].set_title(f"{b} (inicio f{fb})")
        axs[0, j].axis("off"); axs[1, j].axis("off")
    plt.suptitle("¿son la misma persona?"); plt.show()

## 4. A metros: homografía + cinemática

In [ ]:
if IMG_PTS and WORLD_PTS:
    Hm, _ = cv2.findHomography(np.float32(IMG_PTS), np.float32(WORLD_PTS), cv2.RANSAC, 5.0)
    feet = np.c_[(final[:, 2] + final[:, 4]) / 2, final[:, 5]]
    p = np.c_[feet, np.ones(len(feet))] @ Hm.T
    world = p[:, :2] / p[:, 2:3]
else:
    Hm, world = None, None
    print("⚠️ Sin homografía: la analítica se queda en píxeles. Rellena IMG_PTS / WORLD_PTS.")

tracks_df = pd.DataFrame(final[:, :6], columns=["frame", "track_id", "x1", "y1", "x2", "y2"]).astype({"frame": int, "track_id": int})
if world is not None:
    tracks_df["X"], tracks_df["Y"] = world[:, 0], world[:, 1]
tracks_df["interpolated"] = ~pd.MultiIndex.from_frame(tracks_df[["frame", "track_id"]]).isin(
    pd.MultiIndex.from_arrays([stitched[:, 0].astype(int), stitched[:, 1].astype(int)]))

kin, summary = {}, {}
if world is not None:
    for tid, g in tracks_df.groupby("track_id"):
        g = g.set_index("frame").reindex(range(g.frame.min(), g.frame.max() + 1))
        k = kinematics(g[["X", "Y"]].values, fps=FPS, window=int(FPS) | 1)
        k["frame"] = g.index.values
        kin[tid] = k
        if len(k) >= FPS:            # resumen solo para tracks de ≥ 1 s
            summary[tid] = {"duracion_s": round(len(k) / FPS, 1), **player_summary(k, fps=FPS)}
    summary = pd.DataFrame(summary).T
    if len(summary):
        summary = summary.sort_values("duracion_s", ascending=False)
    display(summary.head(15))

In [ ]:
if world is not None:
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    for tid, k in list(kin.items()):
        if len(k) > FPS: ax[0].plot(k.x, k.y, lw=1)
    ax[0].set_xlim(0, WORLD_SIZE[0]); ax[0].set_ylim(WORLD_SIZE[1], 0); ax[0].set_aspect("equal"); ax[0].set_title("trayectorias (m)")
    allk = pd.concat(kin.values())
    ax[1].hist2d(allk.x, allk.y, bins=[24, 15], range=[[0, WORLD_SIZE[0]], [0, WORLD_SIZE[1]]], cmap="hot")
    ax[1].invert_yaxis(); ax[1].set_title("mapa de ocupación"); plt.show()
    v = allk.speed_kmh
    print(f"velocidad mediana {v.median():.1f} km/h (andar ≈ 4–5 km/h → sirve para validar la homografía a ojo)")

## 5. Interacciones: grupos de personas que caminan juntas
Eventos de proximidad (< 1.5 m durante ≥ 2 s) con histéresis, en metros.

In [ ]:
events = []
if world is not None:
    frames = sorted(tracks_df.frame.unique())
    pos = {tid: k.set_index("frame")[["x", "y"]] for tid, k in kin.items() if len(k) > 2 * FPS}
    tids = list(pos)
    for i in range(len(tids)):
        for j in range(i + 1, len(tids)):
            a, b = pos[tids[i]], pos[tids[j]]
            common = a.index.intersection(b.index)
            if len(common) < 2 * FPS: continue
            d = np.linalg.norm(a.loc[common].values - b.loc[common].values, axis=1)
            for s, e in detect_events(d, enter=1.5, exit=2.0, min_len=int(2 * FPS)):
                events.append({"tipo": "proximidad", "ids": [int(tids[i]), int(tids[j])],
                               "t_inicio": round(common[s] / FPS, 2), "t_fin": round(common[e - 1] / FPS, 2),
                               "dist_media_m": round(float(d[s:e].mean()), 2)})
print(len(events), "eventos de proximidad"); pd.DataFrame(events).head(10)

## 6. Vídeo de salida con IDs, rastro y minimapa

In [ ]:
def draw_minimap(world_rows, size=(360, 230)):
    mw, mh = size
    mm = np.full((mh, mw, 3), (40, 110, 40), np.uint8)
    cv2.rectangle(mm, (5, 5), (mw - 6, mh - 6), (255, 255, 255), 2)
    sx, sy = (mw - 10) / WORLD_SIZE[0], (mh - 10) / WORLD_SIZE[1]
    for tid, X, Y in world_rows:
        c = color(tid)
        cv2.circle(mm, (int(5 + X * sx), int(5 + Y * sy)), 5, c, -1)
    return mm

def color(tid):
    return tuple(int(v) for v in np.random.default_rng(int(tid)).integers(60, 255, 3))

out_path = OUT / "resultado.mp4"
cap = cv2.VideoCapture(str(VIDEO))
vw = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
trail = defaultdict(list)
by_frame = {fr: g for fr, g in tracks_df.groupby("frame")}
for fr in range(1, n_frames + 1):
    ok, img = cap.read()
    if not ok: break
    g = by_frame.get(fr)
    if g is not None:
        for _, rr in g.iterrows():
            c = color(rr.track_id)
            cv2.rectangle(img, (int(rr.x1), int(rr.y1)), (int(rr.x2), int(rr.y2)), c, 1 if rr.interpolated else 2)
            cv2.putText(img, str(int(rr.track_id)), (int(rr.x1), int(rr.y1) - 5), 0, 0.7, c, 2)
            trail[rr.track_id].append((int((rr.x1 + rr.x2) / 2), int(rr.y2)))
            pts = trail[rr.track_id][-40:]
            for p, q in zip(pts[:-1], pts[1:]): cv2.line(img, p, q, c, 2)
        if world is not None:
            mm = draw_minimap(g[["track_id", "X", "Y"]].values)
            img[10:10 + mm.shape[0], W - 10 - mm.shape[1]:W - 10] = mm
    vw.write(img)
cap.release(); vw.release()
plt.figure(figsize=(16, 9)); plt.imshow(img[:, :, ::-1]); plt.axis("off")
print("vídeo:", out_path)

## 7. Exportar datos estructurados

In [ ]:
tracks_df.to_csv(OUT / "tracking.csv", index=False)
try:
    tracks_df.to_parquet(OUT / "tracking.parquet", index=False)
except ImportError:
    print("pip install pyarrow para Parquet")
doc = {
    "schema_version": "1.0", "video": VIDEO.name, "fps": FPS, "n_frames": n_frames, "resolution": [W, H],
    "pipeline": {"detector": WEIGHTS.name, "imgsz": IMGSZ, "tracker": "botsort (custom)", "offline": ["min_len", "stitch_reid", "interpolate"],
                 "homography": "manual" if Hm is not None else None},
    "coordinate_system": {"units": "m", "size": WORLD_SIZE} if Hm is not None else {"units": "px"},
    "n_tracks": int(tracks_df.track_id.nunique()),
    "tracks": (summary.reset_index(names="track_id").to_dict(orient="records") if len(summary) else []),
    "events": events,
}
(OUT / "summary.json").write_text(json.dumps(doc, indent=2, ensure_ascii=False, default=float))
print(json.dumps({k: v for k, v in doc.items() if k not in ("tracks", "events")}, indent=2, default=float))

## 8. Evaluación con ground truth (hazlo si te da tiempo: es lo que más impresiona)

1. Sube 15–30 s del vídeo a **CVAT** (app.cvat.ai, gratis) → tarea con la etiqueta `person`.
2. Anota con **tracks** (tecla `N` para crear, keyframes cada ~10 frames, CVAT interpola; marca *outside* cuando
   la persona sale). Usa la auto-anotación con un modelo YOLO si está disponible para ir más rápido.
3. Exporta en formato **MOT 1.1** → copia `gt/gt.txt` a `data/proyecto/gt.txt`.
4. Ejecuta la celda: compara online vs offline con HOTA / IDF1 / MOTA.

In [ ]:
gt_path = OUT / "gt.txt"
if gt_path.exists():
    g = np.loadtxt(gt_path, delimiter=",")
    g = g[(g[:, 6] != 0)] if g.shape[1] > 6 else g                  # conf=0 → ignorar
    gt = np.c_[g[:, 0], g[:, 1], g[:, 2], g[:, 3], g[:, 2] + g[:, 4], g[:, 3] + g[:, 5]]
    last = gt[:, 0].max()
    sel = lambda m: m[m[:, 0] <= last]
    display(pd.DataFrame([evaluate(gt, sel(raw), "online (BoT-SORT)"),
                          evaluate(gt, sel(clean), "+ limpieza"),
                          evaluate(gt, sel(stitched), "+ stitching"),
                          evaluate(gt, sel(final), "+ interpolación")]).round(3))
else:
    print("No hay GT todavía (data/proyecto/gt.txt). Sin GT, las métricas proxy son:")
    print(f"  IDs online: {len(np.unique(raw[:, 1]))} | finales: {tracks_df.track_id.nunique()} | "
          f"personas por frame (mediana): {int(per_frame.median())}")

## 9. Siguientes pasos
- **Precisión**: fine-tuning del detector con frames de tu dominio (notebook 05), modelo `s`/`m`, imgsz mayor
  o tiling para lejanos, ReID entrenado (OSNet) en vez de ImageNet, OC-SORT/Deep OC-SORT si hay movimiento
  brusco, asociación en coordenadas del campo.
- **Identidad en deporte**: clasificación de equipo (k-means de color, notebook 07) como restricción del
  stitching; OCR de dorsales votado por tracklet.
- **Velocidad**: exportar a TensorRT FP16, lector en hilo, ReID en batch y solo cuando hace falta, detección
  cada 2 frames.
- **Producto**: empaquetarlo en la API del notebook 11, cola de trabajos, Parquet a S3, dashboard.